# Codex Python SDK — Researcher Notebook

A practical, cell-based walkthrough of different SDK capabilities.
No test harness, no helper-function-heavy structure.


In [ ]:
# Cell 1: bootstrap current kernel with local SDK + dependencies
import sys, subprocess
from pathlib import Path

repo_python_dir = Path.cwd().resolve().parent  # notebooks/ -> sdk/python
src_dir = repo_python_dir / 'src'

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(repo_python_dir)])
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

print('Kernel:', sys.executable)
print('SDK source:', src_dir)


In [ ]:
# Cell 2: imports
from codex_app_server import Codex, TextInput, ImageInput, LocalImageInput
from codex_app_server.retry import retry_on_overload
from codex_app_server.async_client import AsyncAppServerClient


In [ ]:
# Cell 3: simple sync conversation
with Codex() as codex:
    thread = codex.thread_start(model='gpt-5')
    result = thread.turn(TextInput('Explain gradient descent in 3 bullets.')).run()
    print('status:', result.status)
    print(result.text)


In [ ]:
# Cell 4: multi-turn continuity in same thread
with Codex() as codex:
    thread = codex.thread_start(model='gpt-5')

    first = thread.turn(TextInput('Give a short summary of transformers.')).run()
    second = thread.turn(TextInput('Now explain that to a high-school student.')).run()

    print('first status:', first.status)
    print('second status:', second.status)
    print('second text:', second.text)


In [ ]:
# Cell 5: resume an existing thread by id
with Codex() as codex:
    thread = codex.thread_start(model='gpt-5')
    first = thread.turn(TextInput('Tell me one fact about Saturn.')).run()

    resumed = codex.thread(first.thread_id)
    second = resumed.turn(TextInput('Give one more fact, different from before.')).run()

    print('thread_id:', first.thread_id)
    print('resumed status:', second.status)
    print(second.text)


In [ ]:
# Cell 6: robust request with retry-on-overload
with Codex() as codex:
    thread = codex.thread_start(model='gpt-5')

    result = retry_on_overload(
        lambda: thread.turn(TextInput('List 5 failure modes in distributed systems.')).run(),
        max_attempts=3,
        initial_delay_s=0.25,
        max_delay_s=2.0,
    )

    print('status:', result.status)
    print(result.text)


In [ ]:
# Cell 7: thread lifecycle APIs
with Codex() as codex:
    t = codex.thread_start(model='gpt-5')

    listing = codex.thread_list(limit=10)
    reading = codex.thread_read(t.id, include_turns=False)

    print('thread list fetched')
    print('thread read fetched for:', t.id)


In [ ]:
# Cell 8: multimodal with remote image (best effort)
with Codex() as codex:
    t = codex.thread_start(model='gpt-5')
    try:
        r = t.turn([
            TextInput('What do you see in this image? 3 bullets.'),
            ImageInput('https://upload.wikimedia.org/wikipedia/commons/3/3a/Cat03.jpg'),
        ]).run()
        print('status:', r.status)
        print(r.text)
    except Exception as e:
        print('Remote image path failed in this environment:', type(e).__name__, e)


In [ ]:
# Cell 9: multimodal with local image
from base64 import b64decode
from pathlib import Path

sample_img = Path.cwd() / 'sample.png'
if not sample_img.exists():
    sample_img.write_bytes(
        b64decode('iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mP8/x8AAwMCAO7Z4xQAAAAASUVORK5CYII=')
    )

with Codex() as codex:
    t = codex.thread_start(model='gpt-5')
    r = t.turn([
        TextInput('Describe this local image in 2 bullets.'),
        LocalImageInput(str(sample_img)),
    ]).run()
    print('status:', r.status)
    print(r.text)


In [ ]:
# Cell 10: async usage pattern
import asyncio

async def async_demo():
    async with AsyncAppServerClient() as client:
        await client.initialize()
        started = await client.thread_start(model='gpt-5')
        tid = started['thread']['id']

        turn = await client.turn_text(tid, 'One sentence about scientific reproducibility.')
        turn_id = turn['turn']['id']

        chunks = []
        while True:
            evt = await client.next_notification()
            if evt.method == 'item/agentMessage/delta':
                chunks.append((evt.params or {}).get('delta', ''))
            if evt.method == 'turn/completed' and (evt.params or {}).get('turn', {}).get('id') == turn_id:
                break

        print(''.join(chunks).strip())

await async_demo()


In [ ]:
# Cell 11: explicit error handling example
from codex_app_server.errors import JsonRpcError, MethodNotFoundError, InvalidParamsError, ServerBusyError

with Codex() as codex:
    try:
        codex._client.request('demo/missingMethod', {})  # low-level example
    except MethodNotFoundError as e:
        print('MethodNotFoundError:', e)
    except InvalidParamsError as e:
        print('InvalidParamsError:', e)
    except ServerBusyError as e:
        print('ServerBusyError:', e)
    except JsonRpcError as e:
        print(f'JsonRpcError {e.code}:', e)
